# `alpaca_executor.py` — Playground

Manual verification notebook for the **Alpaca execution layer**: live account state (portfolio value, open positions, daily P&L, drawdown) and order placement.

| Function | Status | Notes |
|---|---|---|
| `get_portfolio_value()` | ✅ built | Total account equity (cash + positions) |
| `get_open_positions()` | ✅ built | Open positions with qty / market value / unrealized P&L |
| `get_daily_pnl()` | ✅ built | Today's P&L (equity − last_equity) |
| `get_drawdown_pct()` | ✅ built | Drawdown from peak equity over the lookback period |
| `position_trade()` | ✅ built | Entry order + native trailing stop; returns an order audit |

**Exit design (Phase 6).** Alpaca supports a `trailing_stop` only as a *single* order — it cannot be the stop leg of a bracket — so a trade is **two orders**: a market entry, then a standalone trailing stop once the entry fills. Alpaca ratchets the stop against the high-water mark broker-side, so there is no monitoring loop.

The trail is **derived**, not a config dial: `trail_percent = trade_levels['stop_pct'] × 100`. That is the same ATR stop distance Gate 5's EV and the risk gate's Kelly sizing were computed from, so the order placed in the market describes the trade that was actually authorised.


In [1]:
import sys
import pathlib

exec_dir = pathlib.Path('.').resolve()
if not (exec_dir / 'alpaca_executor.py').exists():
    exec_dir = pathlib.Path('backend/04_execution').resolve()

if str(exec_dir) not in sys.path:
    sys.path.insert(0, str(exec_dir))

from alpaca_executor import (
    get_portfolio_value,
    get_open_positions,
    get_daily_pnl,
    get_drawdown_pct,
    position_trade,
    _get_alpaca_client,
)


---
## Happy path — live paper account state

Requires `ALPACA_API_KEY` / `ALPACA_SECRET_KEY` / `ALPACA_BASE_URL` in `.env`.


In [2]:
get_portfolio_value()

100000.58

In [3]:
get_open_positions()

[]

In [4]:
get_daily_pnl()

0.5800000000017462

In [5]:
get_drawdown_pct()

0.0

---
## Parameter variation — how the trail is derived

No orders are placed here. This shows why a *fixed* `TRAIL_PERCENT = 1.5` was rejected: the universe filter admits stocks with ATR between 1% and 5% of price, so the ATR stop the position was sized on ranges from **1.5% to 7.5%**. A flat 1.5% trail matches only the calmest name — everything else would be stopped out far tighter than its own math assumed.


In [6]:
import sys, pathlib

# exec_dir was resolved in the setup cell -> backend/04_execution; project root is two up.
root = pathlib.Path(exec_dir).parents[1]
sys.path.insert(0, str(root / 'backend'))
sys.path.insert(0, str(root / 'backend' / '02_intelligence'))
from helpers.logic.trade_levels import build_trade_levels

for label, price, atr in [
    ('low-vol  (ATR 1%)',   100.00,  1.00),
    ('mid-vol  (ATR 2.5%)', 100.00,  2.50),
    ('high-vol (ATR 5%)',   100.00,  5.00),
    ('NVDA reference',      875.50, 12.30),
]:
    levels = build_trade_levels({'price': price, 'atr': atr})
    trail = round(levels['stop_pct'] * 100, 2)   # fraction -> percent, what Alpaca wants
    print(f"{label:22} stop_pct={levels['stop_pct']:.4f}  ->  trail_percent={trail}%")


low-vol  (ATR 1%)      stop_pct=0.0150  ->  trail_percent=1.5%
mid-vol  (ATR 2.5%)    stop_pct=0.0375  ->  trail_percent=3.75%
high-vol (ATR 5%)      stop_pct=0.0750  ->  trail_percent=7.5%
NVDA reference         stop_pct=0.0211  ->  trail_percent=2.11%


---
## Happy path — position a live paper trade

Places a **real paper order**: 1 share of AAPL plus a trailing stop, then cleans up after itself.

Only runs when the market is open — a market order placed while closed just queues, which proves nothing.


In [7]:
api = _get_alpaca_client()

if api is not None and api.get_clock().is_open:
    audit = position_trade({
        'ticker': 'AAPL',
        'shares': 1,
        'trade_levels': {'stop_pct': 0.0211},   # the NVDA reference candidate -> a 2.11% trail
    })
    audit
else:
    audit = None
    print('Market closed — skipping. Next open:', api.get_clock().next_open if api else 'no client')


Market closed — skipping. Next open: 2026-07-14 09:30:00-04:00


In [8]:
# What Alpaca is now holding for us, broker-side
if audit and audit['stop_attached']:
    for o in api.list_orders(status='open'):
        print(f'{o.symbol}  {o.type}  {o.side}  qty={o.qty}  trail={o.trail_percent}%  '
              f'hwm={o.hwm}  stop={o.stop_price}')
    print()
    print('stop_price should equal hwm x (1 - trail/100):',
          round(float(audit['hwm']) * (1 - audit['trail_percent'] / 100), 2))


In [9]:
# Clean up — cancel the trailing stop and close the test position
if audit:
    api.cancel_all_orders()
    api.close_position(audit['ticker'])
    print('cleaned up', audit['ticker'])


---
## Failure path — nothing to place

`shares = 0` must return `None` **before** any order is sent.

Note the contract: `None` always means *no position was opened*. If the entry filled but the trailing stop failed to attach, `position_trade()` returns the audit with `stop_attached: False` instead — returning `None` there would hide a live, unprotected position from the caller.


In [10]:
nothing = position_trade({'ticker': 'AAPL', 'shares': 0, 'trade_levels': {'stop_pct': 0.0211}})
assert nothing is None
print('Zero shares correctly returned None ✅')


[executor] AAPL: nothing to place — shares=0, stop_pct=0.0211
Zero shares correctly returned None ✅


---
## Failure path — missing/invalid credentials

Bad credentials should return `None`, not raise.


In [11]:
import os

good_key = os.environ.get('ALPACA_API_KEY')
os.environ['ALPACA_API_KEY'] = 'invalid'

result = get_portfolio_value()
assert result is None
print('Invalid credentials correctly returned None ✅')

if good_key:
    os.environ['ALPACA_API_KEY'] = good_key


[executor] get_portfolio_value failed: unauthorized.
Invalid credentials correctly returned None ✅


---
## Free-play
